
                         GENERIC BRONZE FRAMEWORK
                                  │
                 ┌────────────────┴─────────────────┐
                 │                                  │
           Configuration                         Functions
                 │                                  │
      source_path / target_table              read_source()
      file_format / load_type                 validate_schema()
      source_name                             validate_data()
                                               transform()
                                               add_metadata()
                                               write_delta()
                                               audit_metrics()
                 │                                  │
                 └────────────────┬─────────────────┘
                                  │
              ┌───────────────────┼────────────────────┐
              │                   │                    │
           Claims              Members              Providers
              │                   │                    │
       bronze.claims       bronze.members       bronze.providers

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import dataframe
from pyspark.sql.types import (StructType, StructField, StringType, IntegerType, LongType, DecimalType, DateType)


In [0]:
# ============================================================
# CONFIGURATION
# ============================================================

source_path = ("abfss://source@mantristorageaccount.dfs.core.windows.net/claims/")
Target_Table = "healthclams.bronze.claims"
File_Formate = "csv"
Source_Name = "Claims"
Load_Type = "Full"
Write_Mode = "overwrite"

In [0]:
# ============================================================
# CLAIMS SCHEMA
# ============================================================

Claims_Schema = StructType([
     StructField("ClaimID", LongType(), False),
    StructField("MemberID", StringType(), False),
    StructField("ProviderID", StringType(), False),
    StructField("ClaimAmount", DecimalType(18, 2), False),
    StructField("ClaimDate", DateType(), False),
    StructField("Status", StringType(), False)
    ])

In [0]:
# ============================================================
# READ SOURCE DATA
# ============================================================

def read_source(
    source_path: str,
    file_format: str,
    schema: StructType
) -> 'DataFrame':
    """
        Reads source data from ADLS.

        Spark concepts:
        - spark.read returns a DataFrameReader.
        - DataFrameReader supports CSV, JSON, Parquet, etc.
        - Explicit schemas improve data consistency.
        - load() creates the DataFrame logical plan.
    """
    try:
        df = (
            spark.read
            .format(file_format)
            .schema(schema)
            .option("inferSchema", True)
            .option("header", True)
            .option("mode", "PERMISSIVE")
            .load(source_path)
        )
        df.printSchema()
        df.show(5, truncate=False)   # <-- sample rows inside function
        print(f"Read {df.count()} records from {source_path}")
        return df
    except Exception as e:
        raise RuntimeError(
            f"Failed to read source data from {source_path}: {str(e)}"
        )

        
# Fixed indentation, function name, argument names, and type hint

In [0]:
df = read_source(source_path, "csv", Claims_Schema)

df.show(5, truncate=False)

In [0]:
# ============================================================
# SCHEMA VALIDATION
# ============================================================

def validate_schema(
    df: dataframe,
    expected_schema: StructType
) -> None:

    """
    Validates the DataFrame schema against the expected schema.

    Spark concept:
    DataFrame.schema returns the StructType representing
    the DataFrame structure.
    """

    actual_schema = df.schema

    if actual_schema != expected_schema:

        print("Expected Schema:")
        expected_schema.simpleString()

        print("Actual Schema:")
        actual_schema.simpleString()

        raise ValueError(
            "Schema validation failed. "
            "Source schema does not match expected schema."
        )

    print("Schema validation successful.")

In [0]:
#df = validate_schema(df, Claims_Schema)
print(Claims_Schema.simpleString())
print(df.schema)
display(df)

In [0]:
# ============================================================
# NULL VALIDATION
# ============================================================

def validate_nulls(
    df: dataframe,
    required_columns: list
) -> None:

    """
    Checks mandatory columns for NULL values.

    Spark concepts:
    - F.col() references a DataFrame column.
    - filter() is a transformation.
    - count() is an action and triggers Spark execution.
    """

    for column_name in required_columns:

        null_count = (
            df
            .filter(F.col(column_name).isNull())
            .count()
        )

        if null_count > 0:

            raise ValueError(
                f"Column '{column_name}' contains "
                f"{null_count} NULL records."
            )

    print("NULL validation successful.")

In [0]:
REQUIRED_COLUMNS = [
    "ClaimID",
    "MemberID",
    "ProviderID",
    "ClaimAmount",
    "ClaimDate",
    "Status"
]

In [0]:
# ============================================================
# DUPLICATE VALIDATION
# ============================================================

def validate_duplicates(
    df: dataframe,
    key_columns: list
) -> None:

    """
    Checks whether duplicate business keys exist.

    Spark concepts:
    - groupBy() creates grouped data.
    - count() calculates records per group.
    - filter() identifies groups with count > 1.
    """

    duplicate_df = (
        df
        .groupBy(*key_columns)
        .count()
        .filter(F.col("count") > 1)
    )

    duplicate_count = duplicate_df.count()

    if duplicate_count > 0:

        raise ValueError(
            f"Duplicate records found for key: "
            f"{key_columns}. "
            f"Duplicate groups: {duplicate_count}"
        )

    print(
        f"Duplicate validation successful for {key_columns}."
    )

In [0]:
validate_duplicates(
    claims_df,
    ["ClaimID"]
)

In [0]:
# ============================================================
# BUSINESS VALIDATION
# ============================================================

def validate_claim_amount(df: dataframe) -> None:

    """
    Validates ClaimAmount.

    Business rule:
    ClaimAmount must be greater than zero.
    """

    invalid_count = (
        df
        .filter(
            (F.col("ClaimAmount").isNull()) |
            (F.col("ClaimAmount") <= 0)
        )
        .count()
    )

    if invalid_count > 0:

        raise ValueError(
            f"Invalid ClaimAmount records found: {invalid_count}"
        )

    print("ClaimAmount validation successful.")

In [0]:
# ============================================================
# STATUS VALIDATION
# ============================================================

VALID_STATUSES = [
    "Approved",
    "Rejected",
    "Pending"
]


def validate_status(df: dataframe) -> None:

    """
    Validates Claims Status against the allowed values.
    """

    invalid_count = (
        df
        .filter(
            ~F.col("Status").isin(VALID_STATUSES)
        )
        .count()
    )

    if invalid_count > 0:

        raise ValueError(
            f"Invalid Status records found: {invalid_count}"
        )

    print("Status validation successful.")

In [0]:
# ============================================================
# GENERIC DATA VALIDATION
# ============================================================

def validate_data(
    df: dataframe,
    expected_schema: StructType,
    required_columns: list,
    key_columns: list
) -> None:

    """
    Executes all common data quality validations.
    """

    validate_schema(
        df,
        expected_schema
    )

    validate_nulls(
        df,
        required_columns
    )

    validate_duplicates(
        df,
        key_columns
    )

    print("Generic data validation completed.")

In [0]:
validate_data(
    claims_df,
    CLAIMS_SCHEMA,
    REQUIRED_COLUMNS,
    ["ClaimID"]
)

validate_claim_amount(claims_df)

validate_status(claims_df)

In [0]:
# ============================================================
# STANDARDIZATION
# ============================================================

def transform_data(df: dataframe) -> dataframe:

    """
    Performs minimal Bronze-layer standardization.

    Bronze principle:
    Preserve source data as much as possible.
    Avoid business transformations in Bronze.
    """

    return (
        df
        .withColumn(
            "Status",
            F.trim(F.col("Status"))
        )
        .withColumn(
            "MemberID",
            F.upper(F.trim(F.col("MemberID")))
        )
        .withColumn(
            "ProviderID",
            F.upper(F.trim(F.col("ProviderID")))
        )
    )


In [0]:
# ============================================================
# INGESTION METADATA
# ============================================================

def add_metadata(df: dataframe) -> dataframe:

    """
    Adds technical metadata required for auditing and lineage.

    input_file_name():
    Returns the source file path for each record.

    current_timestamp():
    Captures the ingestion timestamp.
    """

    return (
        df
        .withColumn(
            "_ingestion_timestamp",
            F.current_timestamp()
        )
        .withColumn(
            "_source_file",
            F.input_file_name()
        )
        .withColumn(
            "_source_system",
            F.lit("ADLS")
        )
    )

In [0]:
# ============================================================
# REJECTED RECORDS
# ============================================================

def split_valid_invalid_claims(
    df: dataframe
):
    """
    Separates valid and invalid Claims records.

    Returns:
        valid_df
        rejected_df
    """

    invalid_condition = (
        F.col("ClaimID").isNull()
        | F.col("MemberID").isNull()
        | F.col("ProviderID").isNull()
        | F.col("ClaimAmount").isNull()
        | (F.col("ClaimAmount") <= 0)
        | F.col("ClaimDate").isNull()
        | ~F.col("Status").isin(VALID_STATUSES)
    )

    rejected_df = df.filter(invalid_condition)

    valid_df = df.filter(~invalid_condition)

    return valid_df, rejected_df

In [0]:
# ============================================================
# DELTA WRITER
# ============================================================

def write_delta(
    df: dataframe,
    target_table: str,
    write_mode: str = "append"
) -> None:

    """
    Writes DataFrame to Delta table in Unity Catalog.

    Delta concepts:
    - Delta is built on Parquet.
    - Delta transaction log provides ACID transactions.
    - saveAsTable() registers/uses the Unity Catalog table.
    """

    if df is None:
        raise ValueError("Cannot write a None DataFrame.")

    (
        df.write
        .format("delta")
        .mode(write_mode)
        .saveAsTable(target_table)
    )

    print(
        f"Successfully written data to {target_table}"
    )

In [0]:
# ============================================================
# AUDIT METRICS
# ============================================================

def generate_audit_metrics(
    source_df: dataframe,
    valid_df: dataframe,
    rejected_df: dataframe
) -> dict:

    """
    Generates basic ingestion metrics.

    Note:
    count() is an action and causes Spark execution.
    """

    source_count = source_df.count()
    valid_count = valid_df.count()
    rejected_count = rejected_df.count()

    metrics = {
        "source_count": source_count,
        "valid_count": valid_count,
        "rejected_count": rejected_count
    }

    return metrics

In [0]:
metrics = generate_audit_metrics(
    claims_df,
    valid_claims_df,
    rejected_claims_df
)

print(metrics)

In [0]:
# ============================================================
# MAIN BRONZE PIPELINE
# ============================================================

try:

    print("=" * 60)
    print("BRONZE INGESTION STARTED")
    print("=" * 60)

    # --------------------------------------------------------
    # Step 1: Read source
    # --------------------------------------------------------

    claims_df = read_source(
        source_path="abfss://source@mantristorageaccount.dfs.core.windows.net/claims/",
        file_format="csv",
        schema=StructType()
    )

    # --------------------------------------------------------
    # Step 2: Validate source
    # --------------------------------------------------------

    # validate_data(
    #     df=claims_df,
    #     expected_schema=CLAIMS_SCHEMA,
    #     required_columns=REQUIRED_COLUMNS,
    #     key_columns=["ClaimID"]
    # )

    # --------------------------------------------------------
    # Step 3: Claims-specific validation
    # --------------------------------------------------------

    # validate_claim_amount(claims_df)

    # validate_status(claims_df)

    # --------------------------------------------------------
    # Step 4: Standardize data
    # --------------------------------------------------------

    # claims_df = transform_data(claims_df)

    # --------------------------------------------------------
    # Step 5: Add metadata
    # --------------------------------------------------------

    #claims_df = add_metadata(claims_df)

    # --------------------------------------------------------
    # Step 6: Split valid/rejected
    # --------------------------------------------------------

    # valid_claims_df, rejected_claims_df = (
    #     split_valid_invalid_claims(claims_df)
    # )

    # --------------------------------------------------------
    # Step 7: Generate audit metrics
    # --------------------------------------------------------

    # metrics = generate_audit_metrics(
    #     claims_df,
    #     valid_claims_df,
    #     rejected_claims_df
    # )

    #print(f"Audit Metrics: {metrics}")

    # --------------------------------------------------------
    # Step 8: Write valid records
    # --------------------------------------------------------

    write_delta(
        claims_df,
        target_table="healthcare.bronze.claims",
        write_mode="append"
    )
    

    # --------------------------------------------------------
    # Step 9: Pipeline completed
    # --------------------------------------------------------

    print("=" * 60)
    print("BRONZE INGESTION COMPLETED SUCCESSFULLY")
    print("=" * 60)

except Exception as e:

    print("=" * 60)
    print("BRONZE INGESTION FAILED")
    print("=" * 60)

    print(f"Error: {str(e)}")

    raise

In [0]:
claims_df = read_source(
        source_path="abfss://source@mantristorageaccount.dfs.core.windows.net/claims/",
        file_format="csv",
        schema=''
    )

claims_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("healthcare.bronze.claims")

display(claims_df)